In [ ]:
import altair as alt
import marimo as mo
import polars as pl

# Loading lives in plain modules, so it can be imported, tested, and
# type-checked without marimo's cell semantics in the way.
from reader import (
    load_details,
    load_report,
    report_outcomes,
    report_tallies,
    unscored_records,
)
from session import find_runs, open_session, summary_table

# Score

What a pipeline found of what was planted, and what it got wrong.

Recall counts a value found under the wrong label, because a redactor
still removes it and nothing leaks; `strict` requires the label to agree,
and the gap between them is what a taxonomy error looks like. Alongside
both, what happened to the values planted to be *ignored* — without which
a pipeline that redacts everything would score perfectly.

For the two sides unmatched, see `explore.py`.

In [ ]:
runs = find_runs()
run_picker = mo.ui.dropdown(
    options={path.name: path for path in runs},
    value=runs[0].name if runs else None,
    label="Run",
)
run_picker if runs else mo.md(
    "**No runs found.** Generate a corpus and run `run` first."
)

&lt;marimo-dropdown data-initial-value=&#x27;[&amp;quot;5-sample&amp;quot;]&#x27; data-label=&#x27;&amp;quot;&amp;#92;u003cspan class=&amp;#92;&amp;quot;markdown prose dark:prose-invert contents&amp;#92;&amp;quot;&amp;#92;u003e&amp;#92;u003cspan class=&amp;#92;&amp;quot;paragraph&amp;#92;&amp;quot;&amp;#92;u003eRun&amp;#92;u003c/span&amp;#92;u003e&amp;#92;u003c/span&amp;#92;u003e&amp;quot;&#x27; data-options=&#x27;[&amp;quot;5-sample&amp;quot;]&#x27; data-allow-select-none=&#x27;false&#x27; data-searchable=&#x27;false&#x27; data-full-width=&#x27;false&#x27; data-disabled=&#x27;false&#x27;&gt;&lt;/marimo-dropdown&gt;

In [ ]:
# Stops here when there is no run, leaving the message above readable
# rather than replacing it with the exception `open_session` would raise.
mo.stop(
    not run_picker.options,
    mo.md("*Nothing to score until a run exists.*"),
)

session = open_session(run_picker.value)
mo.md(summary_table(session))

| | |
|---|---|
| Run | `5-sample` |
| Corpus seed | 5 |
| Pipeline | `benchmark` |
| Records | 12 |
| Labels in scope | 21 |
| Started | 2026-09-08T00:38:58.976Z |
| Scored | yes |

In [ ]:
mo.stop(
    session.report_dir is None,
    mo.md(
        f"""
        /// admonition | Not scored yet
        This run has no report. Write one:

        ```bash
        synthetic score --corpus ./corpus --run {session.run_path} --report ./report
        ```
        ///
        """
    ),
)

# Only the summary is needed for the headline numbers; the per-record
# detail grows with the corpus and is read separately for that reason.
# Rebound so the guard above is visible to a type checker: `mo.stop` ends
# the cell, but nothing in its signature says so.
report_dir = session.report_dir
assert report_dir is not None

report = load_report(report_dir)
details = load_details(report_dir)

## Headline

One line each for the questions the benchmark asks: how much was found,
how much was labelled correctly, how much of each value a redaction would
have covered, and how much of what should have survived did.

In [ ]:
_t = report["totals"]
_located = _t["found"] + _t["mislabelled"]
_cover = _t["boundary"]["covered"] + _t["boundary"]["missed"]

def _pct(numerator: float, denominator: float) -> str:
    return "—" if not denominator else f"{numerator / denominator * 100:.1f}%"

_planted = _t["planted"]
_kept = _t["decoys"] - _t["overRedacted"]

mo.md(
    f"""
    | | |
    |---|---|
    | Found | {_located} of {_planted} — {_pct(_located, _planted)} |
    | Correctly labelled | {_pct(_t["found"], _planted)} |
    | Boundary coverage | {_pct(_t["boundary"]["covered"], _cover)} |
    | Decoys left alone | {_pct(_kept, _t["decoys"])} |
    | Detections on nothing planted | {report["unplanted"]} |
    | Records never processed | {report["failedRecords"]} |
    """
)

| | |
|---|---|
| Found | 77 of 130 — 59.2% |
| Correctly labelled | 59.2% |
| Boundary coverage | 100.0% |
| Decoys left alone | 100.0% |
| Detections on nothing planted | 0 |
| Records never processed | 0 |

In [ ]:
breakdown = mo.ui.dropdown(
    options={
        "label": "byLabel",
        "format": "byFormat",
        "surface form": "bySurface",
        "adversarial kind": "byAdversarial",
    },
    value="label",
    label="Break down by",
)
breakdown

&lt;marimo-dropdown data-initial-value=&#x27;[&amp;quot;label&amp;quot;]&#x27; data-label=&#x27;&amp;quot;&amp;#92;u003cspan class=&amp;#92;&amp;quot;markdown prose dark:prose-invert contents&amp;#92;&amp;quot;&amp;#92;u003e&amp;#92;u003cspan class=&amp;#92;&amp;quot;paragraph&amp;#92;&amp;quot;&amp;#92;u003eBreak down by&amp;#92;u003c/span&amp;#92;u003e&amp;#92;u003c/span&amp;#92;u003e&amp;quot;&#x27; data-options=&#x27;[&amp;quot;label&amp;quot;,&amp;quot;format&amp;quot;,&amp;quot;surface form&amp;quot;,&amp;quot;adversarial kind&amp;quot;]&#x27; data-allow-select-none=&#x27;false&#x27; data-searchable=&#x27;false&#x27; data-full-width=&#x27;false&#x27; data-disabled=&#x27;false&#x27;&gt;&lt;/marimo-dropdown&gt;

In [ ]:
tallies = report_tallies(report, breakdown.value or "byLabel")
tallies.select(
    "key",
    "planted",
    "found",
    "mislabelled",
    "missed",
    pl.col("recall").round(3),
    pl.col("strict").round(3),
    pl.col("coverage").round(3),
    # Found values whose boundary neither side stated a width for, so
    # `coverage` beside it is computed over fewer values than were found.
    "unmeasured",
    "decoys",
    "overRedacted",
)

key,planted,found,mislabelled,missed,recall,strict,coverage,unmeasured,decoys,overRedacted
str,i64,i64,i64,i64,f64,f64,f64,i64,i64,i64
"""person_name""",33,0,0,33,0.0,0.0,null,0,0,0
"""email_address""",18,18,0,0,1.0,1.0,1.0,0,0,0
"""ip_address""",10,10,0,0,1.0,1.0,1.0,0,0,0
"""phone_number""",10,10,0,0,1.0,1.0,1.0,0,1,0
"""payment_card""",9,9,0,0,1.0,1.0,1.0,0,1,0
…,…,…,…,…,…,…,…,…,…,…
"""device_id""",2,0,0,2,0.0,0.0,null,0,2,0
"""mac_address""",1,1,0,0,1.0,1.0,1.0,0,0,0
"""insurance_id""",1,0,0,1,0.0,0.0,null,0,0,0


In [ ]:
mo.stop(tallies.height == 0, mo.md("*Nothing planted under this breakdown.*"))

# Only what was planted to be found: a decoy has no recall to plot, and
# including it would draw an empty bar next to a real one.
_data = tallies.filter(pl.col("planted") > 0).with_columns(
    pl.col("recall").fill_null(0),
    pl.col("strict").fill_null(0),
)

mo.ui.altair_chart(
    alt.Chart(_data)
    .transform_fold(["recall", "strict"], as_=["measure", "value"])
    .mark_bar()
    .encode(
        y=alt.Y(
            "key:N",
            sort=alt.EncodingSortField("planted", order="descending"),
            title=None,
        ),
        x=alt.X("value:Q", title="recall", scale=alt.Scale(domain=[0, 1])),
        yOffset="measure:N",
        color=alt.Color("measure:N", title=None),
        tooltip=[
            "key:N",
            "measure:N",
            alt.Tooltip("value:Q", format=".1%"),
            "planted:Q",
        ],
    )
    .properties(height=alt.Step(12))
)

## What was missed

The values a pipeline did not find, and the ones it found under another
name. A run's headline number says how much leaked; this says what.

In [ ]:
outcomes = report_outcomes(details)
outcomes.filter(pl.col("outcome") == "missed").select(
    "record", "format", "label", "surface", "adversarial", "text"
)

record,format,label,surface,adversarial,text
str,str,str,str,str,str
"""rec_0001""","""txt""","""organization_name""","""canonical""",null,"""Price Inc"""
"""rec_0001""","""txt""","""person_name""","""canonical""","""common_word""","""April Case"""
"""rec_0001""","""txt""","""case_number""","""canonical""","""format_collision""","""471-88-2130"""
"""rec_0001""","""txt""","""person_name""","""abbreviated""","""common_word""","""A. Case"""
"""rec_0001""","""txt""","""username""","""canonical""",null,"""Nellie_Wilkinson21"""
…,…,…,…,…,…
"""rec_0012""","""txt""","""device_id""","""canonical""",null,"""055779a7-1569-4009-a912-34b813…"
"""rec_0012""","""txt""","""person_name""","""canonical""","""coreference""","""Ophelia E. Brakus"""
"""rec_0012""","""txt""","""person_name""","""misspelled""","""coreference""","""Ohpelia E. Brakus"""


In [ ]:
_wrong = outcomes.filter(pl.col("outcome") == "mislabelled")
mo.vstack(
    [
        mo.md("**Found under another label**"),
        _wrong.select("record", "label", "reported", "text")
        if _wrong.height
        else mo.md("*None — every value found was labelled correctly.*"),
    ]
)

Found under another label None — every value found was labelled correctly.

In [ ]:
# Over-redaction: a value planted to survive that was redacted anyway. As
# damaging to a document as a miss is to a person, and invisible to any
# measure of recall.
_over = outcomes.filter(pl.col("outcome") == "over-redacted")
mo.vstack(
    [
        mo.md("**Redacted despite being planted to survive**"),
        _over.select("record", "label", "reported", "text")
        if _over.height
        else mo.md("*None — every decoy survived.*"),
    ]
)

Redacted despite being planted to survive None — every decoy survived.

## Boundary

Whether a detection covered the whole value and no more. Measured only
for values that were found, so a miss cannot masquerade as a boundary
problem. `missed` bytes are what a redaction would leave behind;
`spilled` bytes are document it would destroy.

In [ ]:
_imperfect = outcomes.filter(
    (pl.col("boundaryMissed") > 0) | (pl.col("spilled") > 0)
)
_imperfect.select(
    "record", "label", "text", "covered", "boundaryMissed", "spilled"
).sort("boundaryMissed", descending=True) if _imperfect.height else mo.md(
    "*Every value found was covered exactly.*"
)

*Every value found was covered exactly.*

In [ ]:
# Kept apart from the scores rather than folded in: a record that never
# reached the pipeline was not missed, it was never looked at, and counting
# it as a miss would blame detection for a transport failure.
_unscored = unscored_records(details)
mo.vstack(
    [
        mo.md("**Records the pipeline never processed**"),
        _unscored
        if _unscored.height
        else mo.md("*None — every record was scored.*"),
    ]
)

Records the pipeline never processed None — every record was scored.